# EPR Analysis

A Step-by-Step Superconducting Quantum Chip Design Tutorial: from Theory to Simulation

Tutorial Videos: https://www.youtube.com/playlist?list=PLnK6MrIqGXsJF6XLP-1jIBBhsCS5ZQSmR

James Saslow, Shreyan Juvvadi, Hiu Yung Wong*

contact: Hiu-Yung Wong, hiuyung.wong@sjsu.edu

In [1]:
# Importing Packages
import pyEPR as epr
from qiskit_metal import MetalGUI, Dict, open_docs

C:\Users\012738063\AppData\Local\anaconda3\envs\qmetal3\Lib\site-packages\qutip\__init__.py:71: UserWarning: Runtime cython compilation does not work on Python 3.12.
  warnings.warn(


In [2]:
# Specifying path on my local machine, ansys project name, and design name within the ansys project

path = "C:\\Users\\012738063\\Documents\\Ansoft"
project_name = "Project25" 
design_name = "Full_Structure"


In [3]:
# Opening the project in ansys
pinfo = epr.ProjectInfo(project_path = path,
                        project_name = project_name,
                        design_name  = design_name)

INFO 09:51PM [connect_project]: Connecting to Ansys Desktop API...
INFO 09:51PM [load_ansys_project]: 	File path to HFSS project found.
INFO 09:51PM [load_ansys_project]: 	Opened Ansys App
INFO 09:51PM [load_ansys_project]: 	Opened Ansys Desktop v2026.1.2
INFO 09:51PM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/012738063/Documents/Ansoft/
	Project:   Project25
INFO 09:51PM [connect_design]: 	Opened active design
	Design:    Full_Structure [Solution type: Eigenmode]
INFO 09:51PM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 09:51PM [connect]: 	Connected to project "Project25" and design "Full_Structure" 😀 



In [4]:
# Solution type needs to be 'eigenmode'
pinfo.design.solution_type

'Eigenmode'

In [5]:
# Printing Object Names in ansys design
pinfo.get_all_object_names()

['main',
 'sample_holder',
 'prime_cpw_TQ1_path_1',
 'second_cpw_TQ1',
 'trace_cpw_left',
 'trace_cpw_right',
 'b_wire_Q0_bottom',
 'second_cpw_L_brackeb0_bottom',
 'trace_meander_Q0_bottom',
 'trace_bias0_bottom',
 'launch_pad_Transmission_Launch_Pad_L',
 'launch_pad_Transmission_Launch_Pad_R',
 'pad_top_Q0_bottom',
 'b_connector_pad_Q0_bottom',
 'launch_pad_Launch_Pad_0_Bottom',
 'JJ_rect_Lj_Q0_bottom_rect_jj_R',
 'JJ_rect_Lj_Q0_bottom_rect_jj_L',
 'ground_main_plane',
 'JJ_Lj_Q0_bottom_rect_jj_R_',
 'JJ_Lj_Q0_bottom_rect_jj_L_',
 'prime_cpw_TQ1_1',
 'prime_cpw_TQ1']

In [8]:
# Printing all defined variable names in ansys design
pinfo.design.set_variable("Q0_bottom_Lj", "11.1nH")
pinfo.design.set_variable("Q0_bottom_Cj", "8fF")
pinfo.get_all_variables_names()

['Q0_bottom_Lj', 'Q0_bottom_Cj']

In [9]:
pinfo.design.get_setup_names()

('Setup',)

# Delete JJ_rect_Lj_Q0_bottom_rect_jj_L and JJ_Lj_Q0_bottom_rect_jj_L_ in HFSS

In [10]:
# Defining the Junction in pyEPR
pinfo.junctions['j1'] = {'Lj_variable' : 'Q0_bottom_Lj',        # Inductance Variable Name
                         'rect'        : 'JJ_rect_Lj_Q0_bottom_rect_jj_R',  # Name of the rectangle geometry where Lumped RLC is specified
                         'line'        : 'JJ_Lj_Q0_bottom_rect_jj_R_',   # HFSS Polyline that spans the length of the rectangle
                         'length'      : epr.parse_units('0.06mm'),
                         'Cj_variable' : 'Q0_bottom_Cj'}  


pinfo.validate_junction_info()  # Raises an error if something is wrong with junction definition

In [11]:
# # Simulation Parameters

pinfo.setup.min_pass = 6
pinfo.setup.passes = 15    # Maximum Number of passes
pinfo.setup.n_modes = 6     # Specifying number of modes for eigenmode analysis
pinfo.setup.delta_f = 1
pinfo.setup.min_freq = '1.0GHz'

### **Go into HFSS and click 'analyze'**

In [12]:
pinfo.setup.analyze()

INFO 09:52PM [analyze]: Analyzing setup Setup


0

In [13]:
# pinfo.design.optimetrics.solve_setup(pinfo.design.optimetrics.get_setup_names())

# Eigenmode Analysis

In [14]:
eprh = epr.DistributedAnalysis(pinfo) # epr hfss analysis 

Design "Full_Structure" info:
	# eigenmodes    6
	# variations    1


In [15]:
eprh.get_ansys_frequencies_all()

Freq. (GHz)  Quality Factor
variation mode                             
0         0        4.400222             inf
          1        6.842100             inf
          2       12.658799             inf
          3       13.107244             inf
          4       15.300658             inf
          5       15.484251             inf

In [16]:
eprh.hfss_report_full_convergence()

INFO 10:00PM [hfss_report_full_convergence]: Creating report for variation 0
INFO 10:00PM [hfss_report_f_convergence]: Saved convergences to C:\data-pyEPR\Project25\Full_Structure\hfss_eig_f_convergence.csv


<Figure size 1375x375 with 4 Axes>

<Figure size 1375x375 with 4 Axes>

In [17]:
eprh.get_mesh_statistics()

,Unnamed: 0,Num Tets,Min edge length,Max edge length,RMS edge length,Min tet vol,Max tet vol,Mean tet vol,Std Devn (vol)
0,main,91763,0.003428,0.706239,0.094774,1.656880e-09,0.013452,0.000074,0.000418
1,sample_holder,55375,0.003230,0.728724,0.150537,1.967340e-09,0.020082,0.000292,0.001193


# EPR Analysis

In [18]:
my_epr = eprh.do_EPR_analysis()


Variation 0  [1/1]

  Mode 0 at 4.40 GHz   [1/6]
    Calculating ℰ_magnetic,ℰ_electric
       (ℰ_E-ℰ_H)/ℰ_E       ℰ_E       ℰ_H
               98.8%  1.051e-23 1.232e-25

    Calculating junction energy participation ration (EPR)
	method=`line_voltage`. First estimates:
	junction        EPR p_0j   sign s_0j    (p_capacitive)
		Energy fraction (Lj over Lj&Cj)= 98.33%
	j1               1.05929  (+)        0.0179753
		(U_tot_cap-U_tot_ind)/mean=-2.54%

  Mode 1 at 6.84 GHz   [2/6]
    Calculating ℰ_magnetic,ℰ_electric
       (ℰ_E-ℰ_H)/ℰ_E       ℰ_E       ℰ_H
                0.2%  5.186e-25 5.174e-25

    Calculating junction energy participation ration (EPR)
	method=`line_voltage`. First estimates:
	junction        EPR p_1j   sign s_1j    (p_capacitive)
		Energy fraction (Lj over Lj&Cj)= 96.06%
	j1              0.00274549  (+)        0.000112645
		(U_tot_cap-U_tot_ind)/mean=-0.02%

  Mode 2 at 12.66 GHz   [3/6]
    Calculating ℰ_magnetic,ℰ_electric
       (ℰ_E-ℰ_H)/ℰ_E       ℰ_E       ℰ_

# Cross Kerr Calculation

In [19]:
epra = epr.QuantumAnalysis(eprh.data_filename)

WARNING 10:01PM [__init__]: <p>Error: <class 'IndexError'></p>


	 Differences in variations:




In [20]:
sol = epra.analyze_all_variations(cos_trunc = None, fock_trunc = 6)


 . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 
Variation 0



ERROR 10:01PM [_get_participation_normalized]: WARNING: U_tot_cap-U_tot_ind / mean = 198.4% is > 15%.                     
Is the simulation converged? Proceed with caution
  result['f_ND'] = pd.Series(f1_ND)*1E-6  # MHz

ERROR 10:01PM [_get_participation_normalized]: WARNING: U_tot_cap-U_tot_ind / mean = 198.4% is > 15%.                     
Is the simulation converged? Proceed with caution
  result['Q_coupling'] = self.Qm_coupling[variation][self.Qm_coupling[variation].columns[junctions]][modes]#TODO change the columns to junctions

  result['Qs'] = self.Qs[variation][self.PM[variation].columns[junctions]][modes] #TODO change the columns to junctions



Pm_norm=
modes
0        0.950216
1        0.938655
2        0.790494
3        0.775501
4        1.077401
5    43981.616424
dtype: float64

Pm_norm idx =
      j1
0   True
1  False
2  False
3  False
4  False
5  False
*** P (participation matrix, not normlz.)
         j1
0  1.040581
1  0.002745
2  0.000151
3  0.000213
4  0.000133
5  0.000023

*** S (sign-bit matrix)
   s_j1
0    -1
1    -1
2     1
3     1
4     1
5    -1
*** P (participation matrix, normalized.)
      0.99
    0.0027
   0.00015
   0.00021
   0.00013
   2.3e-05

*** Chi matrix O1 PT (MHz)
    Diag is anharmonicity, off diag is full cross-Kerr.
       161     1.39    0.141    0.206    0.151   0.0258
      1.39  0.00299  0.00061 0.000889 0.000651 0.000111
     0.141  0.00061 3.11e-05 9.06e-05 6.63e-05 1.13e-05
     0.206 0.000889 9.06e-05  6.6e-05 9.66e-05 1.65e-05
     0.151 0.000651 6.63e-05 9.66e-05 3.53e-05 1.21e-05
    0.0258 0.000111 1.13e-05 1.65e-05 1.21e-05 1.03e-06

*** Chi matrix ND (MHz) 

*** Frequencies O1 PT 

In [21]:
''' After disconnecting, you can close Ansys in the external window '''
pinfo.disconnect()